Dans ce notebook on réalise le preprocessing de la base de données à partir des conclusions tirées de l'EDA. Afin d’éviter toute forme de sur-ajustement du protocole d’évaluation, les données ont été séparées en trois sous-ensembles disjoints (apprentissage, validation et test), selon un découpage stratifié sur la variable cible. Le jeu de validation est utilisé pour le réglage des hyperparamètres et la calibration des probabilités, tandis que le jeu de test est réservé à l’évaluation finale des performances et à l’analyse des prédictions. Le split La variable duration a été exclue pour éviter toute fuite d’information. Les pseudo-manquants ont été conservés comme modalités informatives. Les variables fortement asymétriques ont été transformées de manière robuste, et la variable pdays a fait l’objet d’un recodage spécifique afin de capturer l’historique de contact sans explosion de dimension. Les variables catégorielles ont été encodées par one-hot encoding.

In [106]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis, pointbiserialr
df = pd.read_csv("bank-full.csv", sep=';')

# 1. Nettoyage conceptuel : 
### Variables à exclure : duration (fuite d’information (connue après l’appel))

In [107]:
df = df.drop(columns=["duration"])

# 2. Cible et split

### 2.1 Encodage de la cible

In [108]:
y = (df["y"] == "yes").astype(int)
X = df.drop(columns=["y"])

### 2.2 Split stratifié : 

In [109]:
from sklearn.model_selection import StratifiedShuffleSplit

sss_1 = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

for train_val_idx, test_idx in sss_1.split(X, y):
    X_train_val = X.iloc[train_val_idx].copy()
    y_train_val = y.iloc[train_val_idx].copy()
    X_test = X.iloc[test_idx].copy()
    y_test = y.iloc[test_idx].copy()

sss_2 = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.25,  # 0.25 × 0.8 = 0.20 total
    random_state=42
)

for train_idx, val_idx in sss_2.split(X_train_val, y_train_val):
    X_train = X_train_val.iloc[train_idx].copy()
    y_train = y_train_val.iloc[train_idx].copy()
    X_val = X_train_val.iloc[val_idx].copy()
    y_val = y_train_val.iloc[val_idx].copy()

# 3. Traitement des pseudo-manquants
### On ne supprime pas les modalités unknown

# 4. Cas spécial : pdays
Constats EDA : 
- pdays = -1 = jamais contacté (81%) 
- Très forte cardinalité
- Corrélé à previous
-> Décision de preprocessing : Créer 2 variables à partir de pdays : Indicateur binaire : déjà contacté ou pas ? 
 Version tronquée / binée pour les valeurs positives

In [110]:
"""def preprocess_pdays(df):
    df = df.copy()
    df["pdays_contacted"] = (df["pdays"] != -1).astype(int)
    df["pdays_positive"] = df["pdays"].clip(lower=0)
    df = df.drop(columns=["pdays"])
    return df
X_train = preprocess_pdays(X_train)
X_val   = preprocess_pdays(X_val) 
X_test  = preprocess_pdays(X_test)"""

'def preprocess_pdays(df):\n    df = df.copy()\n    df["pdays_contacted"] = (df["pdays"] != -1).astype(int)\n    df["pdays_positive"] = df["pdays"].clip(lower=0)\n    df = df.drop(columns=["pdays"])\n    return df\nX_train = preprocess_pdays(X_train)\nX_val   = preprocess_pdays(X_val) \nX_test  = preprocess_pdays(X_test)'

In [111]:
"""print("pdays_contacted unique:", sorted(X_train["pdays_contacted"].unique()))
print("pdays_positive min:", X_train["pdays_positive"].min())

assert set(X_train["pdays_contacted"].unique()).issubset({0, 1})
assert X_train["pdays_positive"].min() >= 0"""


'print("pdays_contacted unique:", sorted(X_train["pdays_contacted"].unique()))\nprint("pdays_positive min:", X_train["pdays_positive"].min())\n\nassert set(X_train["pdays_contacted"].unique()).issubset({0, 1})\nassert X_train["pdays_positive"].min() >= 0'

# 5. Variables très asymétriques (numériques)
Constats EDA : balance, campaign, previous → longues queues 

Décision : transformation log1p 

La variable balance présente des valeurs positives et négatives, une transformation logarithmique signée de type sign(x)·log(1+|x|) a été utilisée afin de stabiliser la variance tout en conservant l’information de signe.

In [112]:
"""def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))

skewed_vars = ["balance", "campaign", "previous"]

for col in skewed_vars:
    X_train[col] = signed_log1p(X_train[col])
    X_val[col] = signed_log1p(X_val[col])
    X_test[col]  = signed_log1p(X_test[col])"""

'def signed_log1p(x):\n    return np.sign(x) * np.log1p(np.abs(x))\n\nskewed_vars = ["balance", "campaign", "previous"]\n\nfor col in skewed_vars:\n    X_train[col] = signed_log1p(X_train[col])\n    X_val[col] = signed_log1p(X_val[col])\n    X_test[col]  = signed_log1p(X_test[col])'

In [113]:
"""skewed_vars = ["balance", "campaign", "previous"]

print("NaN counts:\n", X_train[skewed_vars].isna().sum())
print("Inf counts:\n", np.isinf(X_train[skewed_vars]).sum())

assert X_train[skewed_vars].isna().sum().sum() == 0
assert np.isinf(X_train[skewed_vars].to_numpy()).sum() == 0"""


'skewed_vars = ["balance", "campaign", "previous"]\n\nprint("NaN counts:\n", X_train[skewed_vars].isna().sum())\nprint("Inf counts:\n", np.isinf(X_train[skewed_vars]).sum())\n\nassert X_train[skewed_vars].isna().sum().sum() == 0\nassert np.isinf(X_train[skewed_vars].to_numpy()).sum() == 0'

# 6. Encodage des variables catégorielles 
- Beaucoup de variables catégorielles
- Signal principalement porté par elles
- Cardinalité modérée

Décision : One-Hot Encoding permet de capturer des effets non linéaires et interactions implicites

In [114]:
cols = ["job", "marital", "education", "contact", "month", "poutcome"]
for c in cols:
    print(c, X_train[c].nunique(), sorted(X_train[c].unique())[:10])

job 12 ['admin.', 'blue-collar', 'entrepreneur', 'housemaid', 'management', 'retired', 'self-employed', 'services', 'student', 'technician']
marital 3 ['divorced', 'married', 'single']
education 4 ['primary', 'secondary', 'tertiary', 'unknown']
contact 3 ['cellular', 'telephone', 'unknown']
month 12 ['apr', 'aug', 'dec', 'feb', 'jan', 'jul', 'jun', 'mar', 'may', 'nov']
poutcome 4 ['failure', 'other', 'success', 'unknown']


In [115]:
# Encodage des variables
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer


categorical_nominal = ["job", "marital", "education", "contact", "month", "poutcome"]
categorical_binary  = ["default", "housing", "loan"]

numerical_features = [
    "age", "balance", "day",
    "campaign", "previous",
    "pdays"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat_nom", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_nominal),
        ("cat_bin", OneHotEncoder(drop="if_binary"), categorical_binary)
    ]
)

# 7. Assemblage final 

In [116]:
X_train_final = preprocessor.fit_transform(X_train)
X_val_final   = preprocessor.transform(X_val)
X_test_final  = preprocessor.transform(X_test)

In [117]:
print("Proportion y=1 train:", y_train.mean())
print("Proportion y=1 val:", y_val.mean())
print("Proportion y=1 test :", y_test.mean())
print("Shapes:", X_train_final.shape, X_test_final.shape)

Proportion y=1 train: 0.11697264616972647
Proportion y=1 val: 0.11700951117009512
Proportion y=1 test : 0.11699657193409267
Shapes: (27126, 41) (9043, 41)


In [118]:
feature_names = preprocessor.get_feature_names_out()

print("Nombre de features :", len(feature_names))
feature_names

Nombre de features : 41


array(['num__age', 'num__balance', 'num__day', 'num__campaign',
       'num__previous', 'num__pdays', 'cat_nom__job_blue-collar',
       'cat_nom__job_entrepreneur', 'cat_nom__job_housemaid',
       'cat_nom__job_management', 'cat_nom__job_retired',
       'cat_nom__job_self-employed', 'cat_nom__job_services',
       'cat_nom__job_student', 'cat_nom__job_technician',
       'cat_nom__job_unemployed', 'cat_nom__job_unknown',
       'cat_nom__marital_married', 'cat_nom__marital_single',
       'cat_nom__education_secondary', 'cat_nom__education_tertiary',
       'cat_nom__education_unknown', 'cat_nom__contact_telephone',
       'cat_nom__contact_unknown', 'cat_nom__month_aug',
       'cat_nom__month_dec', 'cat_nom__month_feb', 'cat_nom__month_jan',
       'cat_nom__month_jul', 'cat_nom__month_jun', 'cat_nom__month_mar',
       'cat_nom__month_may', 'cat_nom__month_nov', 'cat_nom__month_oct',
       'cat_nom__month_sep', 'cat_nom__poutcome_other',
       'cat_nom__poutcome_success', 'cat_

In [119]:
# pour retrouver la base preprocessed dans un notebook de modélisation sans refaire le travail : ``
# Créer un dossier artifacts = tout ce que mon code produit et que je peux régénérer
# dans le notebook preprocessing : 

import os
from joblib import dump

os.makedirs("artifacts", exist_ok=True)

# Fit uniquement sur train
preprocessor.fit(X_train)

# Sauvegardes
dump(preprocessor, "artifacts/preprocessor.joblib")
dump((X_train, X_val, X_test, y_train, y_val, y_test), "artifacts/splits_raw.joblib")

['artifacts/splits_raw.joblib']

In [120]:
# dans un notebook modélisation écrire : 
"""
from joblib import load

preprocessor = load("artifacts/preprocessor.joblib")
X_train, X_val, X_test, y_train, y_val, y_test = load("artifacts/splits_raw.joblib")

X_train_final = preprocessor.transform(X_train)
X_val_final   = preprocessor.transform(X_val)
X_test_final  = preprocessor.transform(X_test)
"""

'\nfrom joblib import load\n\npreprocessor = load("artifacts/preprocessor.joblib")\nX_train, X_val, X_test, y_train, y_val, y_test = load("artifacts/splits_raw.joblib")\n\nX_train_final = preprocessor.transform(X_train)\nX_val_final   = preprocessor.transform(X_val)\nX_test_final  = preprocessor.transform(X_test)\n'